In [ ]:
import random
import json
import re
import asyncio
import pandas as pd
import os
from tqdm.asyncio import tqdm_asyncio
from vpei.utils.llm_requests_v3 import *
from vpei.utils.llm_utils import save_model_experimental_results_to_csv
from vpei.common_utils import extract_score
from vpei.common_variables import *
from vpei.epistemic_consistency.active_prompts import EXPERIMENTS
from vpei.epistemic_consistency.experiment_types import carry_out_absolute_experiment
from vpei.epistemic_consistency.experiment_utils import print_absolute_experiment_results

df = pd.read_csv("./data/_OPC_full_dataset.csv")
df.rename(columns={"problem":"math_problem", "solution":"math_proof"}, inplace=True)
df['proof_length'] = df['math_proof'].apply(lambda x: len(x))
df_correct = df[(df['score'] == '[1]') | (df['score'] == '[1 1]')] # correct proofs
df_incorrect = df[(df['score'] == '[0]') | (df['score'] == '[0 0]')] # incorrect proofs
df = pd.concat([df_correct, df_incorrect], axis=0)
df = df[df['proof_length'] >= 4000]
df_correct['score'] = True
df_incorrect['score'] = False
df

In [ ]:
experiment_name = "math_proofs"
system_prompt = EXPERIMENTS[experiment_name]["absolute_experiment"]["system_prompt"]
user_prompt_template = EXPERIMENTS[experiment_name]["absolute_experiment"]["user_prompt_template"]
print(system_prompt)
print("-------------------------------------------------------------------")
print(user_prompt_template)

In [ ]:
# model_name = "gpt-4o-mini"
model_name = "gpt-5-mini"
model_kwargs = adapt_model_kwargs_for_model(model_name, custom_model_kwargs={})
# professor_name = "John Smith"
name = "J.S."
math_problem = df["math_problem"].iloc[10]
math_proof = df["math_proof"].iloc[10]
user_prompt = user_prompt_template.format(name=name, political_attitude="Republican", math_problem=math_problem, math_proof=math_proof)  
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]
response = make_llm_request(model_name, messages, **model_kwargs)
print("Response:", response)

In [ ]:
models = ["gpt-5-mini"]

n = 1
stimuli_factors = ["math_problem", "math_proof"]
additional_variables_from_df_to_save = ['proof_length']
custom_model_kwargs = {}
random_seed = 42
path_to_save_model_outputs = "./absolute_experiment/"


In [ ]:
payloads = await carry_out_absolute_experiment(models=models, df=df, n=n, system_prompt=system_prompt, user_prompt_template=user_prompt_template, stimuli_factors=stimuli_factors, 
                                               additional_variables_from_df_to_save=additional_variables_from_df_to_save, custom_model_kwargs=custom_model_kwargs, 
                                               path_to_save_model_outputs=path_to_save_model_outputs, random_seed=random_seed)

print_absolute_experiment_results(payloads, models)